# 단일 도착지 최저가 직항 찾기

아래 **앱 시작하기** 칸의 ▶ 버튼을 한 번 누르면 앱이 열립니다.

> 실제 결제 전 가격, 수하물, 수수료는 예약 화면에서 다시 확인하세요.

In [ ]:
#@title 앱 시작하기
#@markdown 왼쪽 ▶ 버튼 한 번만 누르면 설치, 실행, 앱 열기까지 한 번에 진행됩니다.

from pathlib import Path
import os
import signal
import socket
import subprocess
import time

REPO_URL = "https://github.com/jjunsss/cheapest-flights.git"
APP_DIR = Path("/content/cheapest-flights")
LOG_PATH = APP_DIR / "colab-app.log"
PID_PATH = Path("/content/flight_app.pid")
SEARCH_WIDTH = 24
PARALLEL_SEARCHES = 2


def run(command, cwd=None):
    print("$", " ".join(command) if isinstance(command, list) else command)
    subprocess.run(command, cwd=cwd, check=True)


def node_major():
    try:
        result = subprocess.run(
            ["node", "-p", "Number(process.versions.node.split('.')[0])"],
            check=True,
            capture_output=True,
            text=True,
        )
        return int(result.stdout.strip())
    except Exception:
        return 0


def stop_previous():
    if PID_PATH.exists():
        try:
            pid = int(PID_PATH.read_text().strip())
            os.kill(pid, signal.SIGTERM)
            time.sleep(1)
        except Exception:
            pass
    subprocess.run("pkill -f 'npm run dev' || true", shell=True)
    subprocess.run("pkill -f 'tsx watch src/server/index.ts' || true", shell=True)
    subprocess.run("pkill -f 'vite --host' || true", shell=True)


def wait_for_port(port, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(1)
            if sock.connect_ex(("127.0.0.1", port)) == 0:
                return True
        time.sleep(1)
    return False


def show_log_tail():
    if not LOG_PATH.exists():
        return
    print("\n--- 실행 로그 마지막 부분 ---")
    lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-80:]))


stop_previous()

if not APP_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, str(APP_DIR)])
else:
    print("최신 버전으로 갱신합니다.")
    run(["git", "fetch", "--depth", "1", "origin", "main"], cwd=APP_DIR)
    run(["git", "reset", "--hard", "origin/main"], cwd=APP_DIR)

os.chdir(APP_DIR)

if node_major() < 20:
    print("Node.js 22를 준비합니다. 잠시만 기다려 주세요.")
    run(["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs"])

run(["npm", "install"], cwd=APP_DIR)
run(["npx", "playwright", "install", "chromium", "--with-deps"], cwd=APP_DIR)

log_file = open(LOG_PATH, "w", encoding="utf-8")
process = subprocess.Popen(
    ["npm", "run", "dev"],
    cwd=APP_DIR,
    env={
        **os.environ,
        "HOST": "127.0.0.1",
        "PORT": "3001",
        "FLIGHT_MAX_PAIRS": str(SEARCH_WIDTH),
        "FLIGHT_PARALLEL": str(PARALLEL_SEARCHES),
        "FLIGHT_BLOCK_HEAVY_RESOURCES": "1",
        "FLIGHT_BLOCK_IMAGES": "0",
    },
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
)
PID_PATH.write_text(str(process.pid))

front_ready = wait_for_port(5173, 90)
back_ready = wait_for_port(3001, 90)

if not (front_ready and back_ready):
    show_log_tail()
    raise RuntimeError("앱 시작에 실패했습니다. 위 로그를 확인해 주세요.")

try:
    from google.colab import output
    print("아래에 앱 화면을 띄웁니다.")
    output.serve_kernel_port_as_iframe(5173, height=900)
except Exception as error:
    show_log_tail()
    print("앱 열기에 실패했습니다:", error)
    print("이 셀을 한 번 더 실행해 주세요.")
